# Reusable ARM97 Experiment Batch Plotting

This notebook plots a batch of E3SM SCM experiment history files against a continuous baseline and, when a clear ARM97 observation mapping exists, against observation as well. It is intended as a reusable plotting template: update the configuration cell for a new experiment directory or filename pattern, then rerun the notebook.

Behavior:

- Variables with mapped observations: plot `observation + baseline + experiment members`; lower panel is `model - observation`.
- Variables without mapped observations: plot `baseline + experiment members`; lower panel is `experiment member - baseline`.
- Numeric variables with `time` as the first dimension are discovered automatically, and extra dimensions are averaged into a time series.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import timedelta
import math
import os
from pathlib import Path
import re


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "e3sm_scm_run_scripts_baseline").exists():
            return candidate
    return Path("/Users/yunlong/Workshop/SCM-UQ-Workflow")


ROOT = Path(os.environ.get("SCM_UQ_WORKFLOW_ROOT", find_repo_root())).resolve()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".local_cache/matplotlib-cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".local_cache"))

import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("repo root:", ROOT)


## Configuration

For a new experiment, usually only change `EXPERIMENT_DIR`, `FILE_GLOB`, `RUN_ID_REGEX`, and `CASE_LABEL`. The defaults point to the current ARM97 qmc14x5 stitched run so the notebook can be executed immediately.


In [ ]:
DEFAULT_BASELINE = (
    ROOT
    / "e3sm_scm_run_scripts_baseline/baseline-output/scm_ARM97_baseline/run"
    / "case_scripts.eam.h0.1997-06-19-84585.nc"
)
DEFAULT_OBSERVATION = ROOT / "e3sm_scm_run_scripts_baseline/ARM97_iopfile_4scam.nc"
DEFAULT_EXPERIMENT_DIR = (
    ROOT
    / "arm97_experiments_0602/arm97_qmc14x5_stitched_seed20260602/mac/stitched"
)

BASELINE = Path(os.environ.get("SCM_BASELINE_HISTORY_FILE", DEFAULT_BASELINE)).expanduser().resolve()
OBSERVATION = Path(os.environ.get("ARM97_IOP_FILE", DEFAULT_OBSERVATION)).expanduser().resolve()
EXPERIMENT_DIR = Path(os.environ.get("SCM_EXPERIMENT_HISTORY_DIR", DEFAULT_EXPERIMENT_DIR)).expanduser().resolve()

# File discovery. RUN_ID_REGEX should contain one capturing group for the run/member id.
FILE_GLOB = os.environ.get("SCM_EXPERIMENT_FILE_GLOB", "mac_ARM97_qmc14x5_*_stitched_26day.nc")
RUN_ID_REGEX = os.environ.get("SCM_EXPERIMENT_RUN_ID_REGEX", r"_(\d{3})_stitched_26day\.nc$")
CASE_LABEL = os.environ.get("SCM_EXPERIMENT_CASE_LABEL", EXPERIMENT_DIR.parent.parent.name)

OUT_DIR = Path(os.environ.get("SCM_EXPERIMENT_PLOT_OUT_DIR", EXPERIMENT_DIR / "batch_figures")).expanduser().resolve()

BATCH_SIZE = 10
BATCH_INDEX = 0
SELECTED_RUN_IDS: list[int | str] | None = None

# Set to a list such as ["TREFHT", "CLDTOT"] to limit output.
# Leave as None to plot every discovered numeric time-dependent variable.
INCLUDE_VARIABLES: list[str] | None = None

EXCLUDE_VARIABLES = {
    "time",
    "time_bnds",
    "date",
    "datesec",
    "date_written",
    "time_written",
    "ndcur",
    "nscur",
    "nsteph",
}

assert BASELINE.exists(), BASELINE
assert OBSERVATION.exists(), OBSERVATION
assert EXPERIMENT_DIR.exists(), EXPERIMENT_DIR

print("baseline:", BASELINE)
print("observation:", OBSERVATION)
print("experiment dir:", EXPERIMENT_DIR)
print("file glob:", FILE_GLOB)
print("run id regex:", RUN_ID_REGEX)
print("case label:", CASE_LABEL)
print("figure output:", OUT_DIR)


## Observation Mapping

These ARM97 mappings are intentionally explicit. If a variable is not listed here, or the observation file does not contain the mapped observation variable, the notebook falls back to baseline-only comparison for that model variable.


In [ ]:
@dataclass(frozen=True)
class VarSpec:
    model: str
    obs: str | None
    units: str
    scale_obs: float = 1.0
    obs_offset: float = 0.0
    description: str = ""


OBSERVED_VAR_SPECS = {
    "TREFHT": VarSpec("TREFHT", "Tsair", "K", description="2 m air temperature"),
    "TS": VarSpec("TS", "Tg", "K", description="surface/ground temperature"),
    "TMQ": VarSpec("TMQ", "prew", "kg/m2", scale_obs=10.0, description="precipitable water"),
    "CLDTOT": VarSpec("CLDTOT", "totcld", "1", scale_obs=0.01, description="total cloud fraction"),
    "CLDLOW": VarSpec("CLDLOW", "lowcld", "1", scale_obs=0.01, description="low cloud fraction"),
    "CLDMED": VarSpec("CLDMED", "midcld", "1", scale_obs=0.01, description="mid-level cloud fraction"),
    "CLDHGH": VarSpec("CLDHGH", "hghcld", "1", scale_obs=0.01, description="high cloud fraction"),
    "PS": VarSpec("PS", "Ps", "Pa", description="surface pressure"),
    "LHFLX": VarSpec("LHFLX", "lhflx", "W/m2", description="latent heat flux"),
    "SHFLX": VarSpec("SHFLX", "shflx", "W/m2", description="sensible heat flux"),
    "FSNS": VarSpec("FSNS", "srfswdn-srfswup", "W/m2", description="surface net shortwave flux"),
    "FLNS": VarSpec("FLNS", "srflwup-srflwdn", "W/m2", description="surface net longwave flux"),
    "FSDS": VarSpec("FSDS", "srfswdn", "W/m2", description="surface downwelling shortwave flux"),
    "FLDS": VarSpec("FLDS", "srflwdn", "W/m2", description="surface downwelling longwave flux"),
    "FLUT": VarSpec("FLUT", "TOA_LWup", "W/m2", description="TOA upwelling longwave flux"),
    "U10": VarSpec("U10", "windsrf", "m/s", description="10 m wind speed"),
    "T": VarSpec("T", "T", "K", description="temperature, averaged over extra dimensions"),
    "Q": VarSpec("Q", "q", "kg/kg", description="specific humidity, averaged over extra dimensions"),
    "U": VarSpec("U", "u", "m/s", description="zonal wind, averaged over extra dimensions"),
    "V": VarSpec("V", "v", "m/s", description="meridional wind, averaged over extra dimensions"),
    "OMEGA": VarSpec("OMEGA", "omega", "Pa/s", description="pressure vertical velocity, averaged over extra dimensions"),
    "RELHUM": VarSpec("RELHUM", "rh", "%", description="relative humidity, averaged over extra dimensions"),
}


## Discover Experiment Files


In [ ]:
def parse_run_id(path: Path, regex: str = RUN_ID_REGEX):
    match = re.search(regex, path.name)
    if not match:
        return None
    value = match.group(1)
    try:
        return int(value)
    except ValueError:
        return value


def sort_key(value):
    if isinstance(value, int):
        return (0, value)
    return (1, str(value))


def discover_experiment_files(experiment_dir: Path = EXPERIMENT_DIR, file_glob: str = FILE_GLOB) -> pd.DataFrame:
    rows = []
    for index, path in enumerate(sorted(experiment_dir.glob(file_glob))):
        run_id = parse_run_id(path)
        if run_id is None:
            run_id = index
        rows.append({"run_id": run_id, "path": path})
    if not rows:
        raise FileNotFoundError(f"No files matched {file_glob!r} in {experiment_dir}")
    df = pd.DataFrame(rows)
    df = df.sort_values("run_id", key=lambda s: s.map(sort_key)).reset_index(drop=True)
    return df


def select_batch(files: pd.DataFrame, batch_size: int, batch_index: int, selected_run_ids=None) -> pd.DataFrame:
    if selected_run_ids is not None:
        selected = files[files["run_id"].isin(selected_run_ids)].copy()
        missing = sorted(set(selected_run_ids) - set(selected["run_id"]), key=sort_key)
        if missing:
            raise ValueError(f"Missing requested run ids: {missing}")
        return selected.sort_values("run_id", key=lambda s: s.map(sort_key)).reset_index(drop=True)

    if batch_size <= 0:
        raise ValueError("BATCH_SIZE must be positive")
    start = batch_index * batch_size
    stop = start + batch_size
    selected = files.iloc[start:stop].copy()
    if selected.empty:
        n_batches = math.ceil(len(files) / batch_size)
        raise ValueError(f"BATCH_INDEX {batch_index} is empty; valid range is 0..{n_batches - 1}")
    return selected.reset_index(drop=True)


def run_label(run_id) -> str:
    return f"{run_id:03d}" if isinstance(run_id, int) else str(run_id)


ALL_FILES = discover_experiment_files()
BATCH_FILES = select_batch(ALL_FILES, BATCH_SIZE, BATCH_INDEX, SELECTED_RUN_IDS)
N_BATCHES = math.ceil(len(ALL_FILES) / BATCH_SIZE)

print(f"found {len(ALL_FILES)} experiment files")
print(f"batch {BATCH_INDEX + 1}/{N_BATCHES}:", [run_label(x) for x in BATCH_FILES["run_id"]])
BATCH_FILES


## Load Data


In [ ]:
def as_series(var):
    data = np.ma.asarray(var[:], dtype=np.float64)
    if data.ndim == 1:
        return data
    axes = tuple(range(1, data.ndim))
    return np.ma.mean(data, axis=axes)


def obs_series(ds, expression: str):
    if "-" in expression:
        left, right = expression.split("-", 1)
        return as_series(ds.variables[left]) - as_series(ds.variables[right])
    return as_series(ds.variables[expression])


def filled(arr):
    return np.asarray(np.ma.asarray(arr, dtype=np.float64).filled(np.nan), dtype=np.float64)


def interpolate_reference(source_days, source_values, target_days):
    finite = np.isfinite(source_values)
    if finite.sum() < 2:
        return np.full_like(target_days, np.nan, dtype=np.float64)
    return np.interp(target_days, source_days[finite], source_values[finite], left=np.nan, right=np.nan)


def score(model_values, reference_values):
    finite = np.isfinite(model_values) & np.isfinite(reference_values)
    if not finite.any():
        return dict(n=0, bias=np.nan, mae=np.nan, rmse=np.nan, max_abs=np.nan)
    diff = model_values[finite] - reference_values[finite]
    return dict(
        n=int(diff.size),
        bias=float(np.mean(diff)),
        mae=float(np.mean(np.abs(diff))),
        rmse=float(np.sqrt(np.mean(diff * diff))),
        max_abs=float(np.max(np.abs(diff))),
    )


def load_time_axis(ds):
    t = ds.variables["time"]
    days = np.asarray(t[:], dtype=np.float64)
    dates = np.array(num2date(days, t.units, getattr(t, "calendar", "standard"), only_use_cftime_datetimes=False))
    return days, dates


def is_numeric_time_var(var) -> bool:
    dims = getattr(var, "dimensions", ())
    if not dims or dims[0] != "time":
        return False
    try:
        return np.issubdtype(np.dtype(var.dtype), np.number)
    except TypeError:
        return False


def obs_expression_available(obs, expression: str | None) -> bool:
    if expression is None:
        return False
    return all(name in obs.variables for name in expression.split("-"))


def discover_variable_specs(batch_files=BATCH_FILES) -> list[VarSpec]:
    sample_path = Path(batch_files.iloc[0]["path"])
    include = set(INCLUDE_VARIABLES) if INCLUDE_VARIABLES is not None else None
    specs = []

    with Dataset(BASELINE) as baseline, Dataset(sample_path) as experiment, Dataset(OBSERVATION) as obs:
        for name, var in baseline.variables.items():
            if name in EXCLUDE_VARIABLES:
                continue
            if include is not None and name not in include:
                continue
            if name not in experiment.variables:
                continue
            if not is_numeric_time_var(var) or not is_numeric_time_var(experiment.variables[name]):
                continue

            if name in OBSERVED_VAR_SPECS and obs_expression_available(obs, OBSERVED_VAR_SPECS[name].obs):
                specs.append(OBSERVED_VAR_SPECS[name])
                continue

            units = getattr(var, "units", "") or ""
            description = getattr(var, "long_name", "") or name
            specs.append(VarSpec(name, None, units, description=description))

    if not specs:
        raise ValueError("No plottable variables discovered")
    return specs


def load_experiment_batch(specs=None, batch_files=BATCH_FILES):
    specs = discover_variable_specs(batch_files) if specs is None else list(specs)
    data = {}
    rows = []

    with Dataset(BASELINE) as baseline, Dataset(OBSERVATION) as obs:
        baseline_days, baseline_dates = load_time_axis(baseline)
        obs_days = (np.asarray(obs.variables["tsec"][:], dtype=np.float64) - float(obs.variables["tsec"][0])) / 86400.0
        origin = baseline_dates[0] - timedelta(days=float(baseline_days[0]))
        obs_dates = np.array([origin + timedelta(days=float(x)) for x in obs_days])

        for spec in specs:
            if spec.model not in baseline.variables:
                print(f"skip {spec.model}: missing baseline variable")
                continue

            has_obs = obs_expression_available(obs, spec.obs)
            baseline_values = filled(as_series(baseline.variables[spec.model]))
            obs_native = None
            obs_at_baseline = None
            baseline_diff_obs = None
            baseline_stats = None
            if has_obs:
                obs_native = filled(obs_series(obs, spec.obs)) * spec.scale_obs + spec.obs_offset
                obs_at_baseline = interpolate_reference(obs_days, obs_native, baseline_days)
                baseline_diff_obs = baseline_values - obs_at_baseline
                baseline_stats = score(baseline_values, obs_at_baseline)

            members = []
            for row in batch_files.itertuples(index=False):
                with Dataset(row.path) as experiment:
                    if spec.model not in experiment.variables:
                        print(f"skip run {run_label(row.run_id)} {spec.model}: missing in experiment file")
                        continue
                    member_days, member_dates = load_time_axis(experiment)
                    values = filled(as_series(experiment.variables[spec.model]))
                    baseline_at_member = interpolate_reference(baseline_days, baseline_values, member_days)
                    member = {
                        "run_id": row.run_id,
                        "path": Path(row.path),
                        "days": member_days,
                        "dates": member_dates,
                        "values": values,
                        "baseline_at_member": baseline_at_member,
                        "diff_baseline": values - baseline_at_member,
                    }
                    if has_obs:
                        obs_at_member = interpolate_reference(obs_days, obs_native, member_days)
                        member["obs_at_member"] = obs_at_member
                        member["diff_obs"] = values - obs_at_member
                    members.append(member)

            if not members:
                continue

            for member in members:
                if has_obs:
                    comparison = "observation"
                    row_stats = score(member["values"], member["obs_at_member"])
                else:
                    comparison = "baseline"
                    row_stats = score(member["values"], member["baseline_at_member"])
                rows.append(
                    {
                        "case_label": CASE_LABEL,
                        "variable": spec.model,
                        "run_id": member["run_id"],
                        "comparison": comparison,
                        "observation": spec.obs if has_obs else "",
                        "description": spec.description,
                        "units": spec.units,
                        **row_stats,
                    }
                )

            data[spec.model] = {
                "spec": spec,
                "has_obs": has_obs,
                "baseline_days": baseline_days,
                "baseline_dates": baseline_dates,
                "obs_days": obs_days,
                "obs_dates": obs_dates,
                "obs_native": obs_native,
                "obs_at_baseline": obs_at_baseline,
                "baseline_values": baseline_values,
                "baseline_diff_obs": baseline_diff_obs,
                "baseline_stats": baseline_stats,
                "members": members,
            }

    summary = pd.DataFrame(rows)
    if not summary.empty:
        summary = summary.sort_values(["comparison", "variable", "rmse", "run_id"]).reset_index(drop=True)
    return data, summary


VARIABLE_SPECS = discover_variable_specs(BATCH_FILES)
DATA, SUMMARY = load_experiment_batch(VARIABLE_SPECS, BATCH_FILES)
with_obs = sum(d["has_obs"] for d in DATA.values())
print(f"discovered {len(VARIABLE_SPECS)} plottable variables")
print(f"loaded {len(DATA)} variables for {len(BATCH_FILES)} selected runs")
print(f"variables with observation: {with_obs}; baseline-only variables: {len(DATA) - with_obs}")
SUMMARY.head(20)


## Interactive Figure

The dropdown switches variables for the currently selected batch. For large variable lists, the static export cell below is often more practical than scrolling through the dropdown.


In [ ]:
def variable_title(name: str, d: dict) -> str:
    spec = d["spec"]
    run_ids = [m["run_id"] for m in d["members"]]
    run_range = f"{run_label(min(run_ids, key=sort_key))}-{run_label(max(run_ids, key=sort_key))}"
    if d["has_obs"]:
        s = d["baseline_stats"]
        return (
            f"{CASE_LABEL} {name}: observation, baseline, and experiment members"
            f"<br><sup>{spec.description} | obs={spec.obs} | runs {run_range} | "
            f"baseline bias={s['bias']:.3g} {spec.units}, RMSE={s['rmse']:.3g}, n={s['n']}</sup>"
        )
    return (
        f"{CASE_LABEL} {name}: baseline and experiment members"
        f"<br><sup>{spec.description} | no mapped observation | runs {run_range} | lower panel is member - baseline</sup>"
    )


def make_interactive_batch_figure(data: dict[str, dict] = DATA) -> go.Figure:
    names = list(data)
    if not names:
        raise ValueError("DATA is empty")

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10, row_heights=[0.68, 0.32])
    trace_counts = []

    for i, name in enumerate(names):
        d = data[name]
        visible = i == 0
        spec = d["spec"]
        start_count = len(fig.data)

        if d["has_obs"]:
            fig.add_trace(
                go.Scatter(
                    x=d["obs_dates"], y=d["obs_native"], mode="lines", name="observation",
                    legendgroup=f"{name}-reference", line=dict(color="rgba(35,35,35,0.78)", width=1.3),
                    visible=visible, hovertemplate="%{x}<br>obs=%{y:.4g}<extra></extra>",
                ),
                row=1, col=1,
            )
        fig.add_trace(
            go.Scatter(
                x=d["baseline_dates"], y=d["baseline_values"], mode="lines", name="baseline",
                legendgroup=f"{name}-reference", line=dict(color="#145DA0", width=2.4),
                visible=visible, hovertemplate="%{x}<br>baseline=%{y:.4g}<extra></extra>",
            ),
            row=1, col=1,
        )

        for member in d["members"]:
            label = run_label(member["run_id"])
            fig.add_trace(
                go.Scatter(
                    x=member["dates"], y=member["values"], mode="lines", name=f"member {label}",
                    legendgroup=f"{name}-members", line=dict(width=1.05), opacity=0.64,
                    visible=visible, hovertemplate=f"%{{x}}<br>member {label}=%{{y:.4g}}<extra></extra>",
                ),
                row=1, col=1,
            )

        if d["has_obs"]:
            fig.add_trace(
                go.Scatter(
                    x=d["baseline_dates"], y=d["baseline_diff_obs"], mode="lines", name="baseline - observation",
                    legendgroup=f"{name}-diff", line=dict(color="#C2410C", width=1.8),
                    visible=visible, hovertemplate="%{x}<br>baseline - obs=%{y:.4g}<extra></extra>",
                ),
                row=2, col=1,
            )
            for member in d["members"]:
                label = run_label(member["run_id"])
                fig.add_trace(
                    go.Scatter(
                        x=member["dates"], y=member["diff_obs"], mode="lines", name=f"member {label} - observation",
                        legendgroup=f"{name}-memberdiff", showlegend=False, line=dict(width=0.9), opacity=0.50,
                        visible=visible, hovertemplate=f"%{{x}}<br>member {label} - obs=%{{y:.4g}}<extra></extra>",
                    ),
                    row=2, col=1,
                )
        else:
            for member in d["members"]:
                label = run_label(member["run_id"])
                fig.add_trace(
                    go.Scatter(
                        x=member["dates"], y=member["diff_baseline"], mode="lines", name=f"member {label} - baseline",
                        legendgroup=f"{name}-memberdiff", showlegend=False, line=dict(width=0.95), opacity=0.55,
                        visible=visible, hovertemplate=f"%{{x}}<br>member {label} - baseline=%{{y:.4g}}<extra></extra>",
                    ),
                    row=2, col=1,
                )

        trace_counts.append(len(fig.data) - start_count)

    buttons = []
    offset = 0
    total = len(fig.data)
    for name, count in zip(names, trace_counts):
        mask = [False] * total
        mask[offset : offset + count] = [True] * count
        d = data[name]
        spec = d["spec"]
        diff_label = "model - observation" if d["has_obs"] else "member - baseline"
        buttons.append(
            dict(
                label=f"{name}: {spec.description}",
                method="update",
                args=[
                    {"visible": mask},
                    {
                        "title.text": variable_title(name, d),
                        "yaxis.title.text": f"{name} ({spec.units})" if spec.units else name,
                        "yaxis2.title.text": f"{diff_label} ({spec.units})" if spec.units else diff_label,
                    },
                ],
            )
        )
        offset += count

    first_name = names[0]
    first = data[first_name]
    first_units = first["spec"].units
    first_diff_label = "model - observation" if first["has_obs"] else "member - baseline"
    fig.update_layout(
        title=dict(text=variable_title(first_name, first), x=0.01, xanchor="left"),
        template="plotly_white", height=800, width=1180, hovermode="x unified",
        margin=dict(l=80, r=40, t=125, b=70),
        legend=dict(orientation="h", yanchor="bottom", y=1.005, xanchor="left", x=0),
        updatemenus=[dict(type="dropdown", x=1.0, y=1.14, xanchor="right", yanchor="top", buttons=buttons, showactive=True)],
    )
    fig.update_xaxes(showticklabels=False, row=1, col=1)
    fig.update_xaxes(title="Time", row=2, col=1)
    fig.update_yaxes(title=f"{first_name} ({first_units})" if first_units else first_name, row=1, col=1)
    fig.update_yaxes(
        title=f"{first_diff_label} ({first_units})" if first_units else first_diff_label,
        zeroline=True, zerolinewidth=1, zerolinecolor="black", row=2, col=1,
    )
    return fig


fig = make_interactive_batch_figure(DATA)
fig.show()


## Static Export

This writes one PNG per discovered variable and one CSV summary for the selected batch.


In [ ]:
import matplotlib.pyplot as plt


def export_batch_pngs(data: dict[str, dict] = DATA, out_dir: Path = OUT_DIR) -> list[Path]:
    batch_label = "custom" if SELECTED_RUN_IDS is not None else f"batch{BATCH_INDEX:02d}"
    fig_dir = out_dir / batch_label
    fig_dir.mkdir(parents=True, exist_ok=True)
    paths = []

    for name, d in data.items():
        spec = d["spec"]
        fig, axes = plt.subplots(
            2, 1, figsize=(12, 7), sharex=True,
            gridspec_kw={"height_ratios": [2.1, 1.0], "hspace": 0.08},
        )
        if d["has_obs"]:
            axes[0].plot(d["obs_dates"], d["obs_native"], color="0.20", lw=1.1, alpha=0.72, label="observation")
        axes[0].plot(d["baseline_dates"], d["baseline_values"], color="#145DA0", lw=2.0, label="baseline")
        for member in d["members"]:
            axes[0].plot(member["dates"], member["values"], lw=0.9, alpha=0.55, label=f"member {run_label(member['run_id'])}")
        axes[0].set_ylabel(f"{name} ({spec.units})" if spec.units else name)
        axes[0].legend(loc="best", frameon=False, ncols=2, fontsize=8)
        axes[0].grid(True, alpha=0.25)

        if d["has_obs"]:
            axes[1].plot(d["baseline_dates"], d["baseline_diff_obs"], color="#C2410C", lw=1.4, label="baseline - observation")
            for member in d["members"]:
                axes[1].plot(member["dates"], member["diff_obs"], lw=0.75, alpha=0.45)
            diff_label = "model - obs"
        else:
            for member in d["members"]:
                axes[1].plot(member["dates"], member["diff_baseline"], lw=0.85, alpha=0.50)
            diff_label = "member - baseline"
        axes[1].axhline(0, color="black", lw=0.8)
        axes[1].set_ylabel(f"{diff_label} ({spec.units})" if spec.units else diff_label)
        axes[1].set_xlabel("Time")
        axes[1].grid(True, alpha=0.25)

        run_ids = [m["run_id"] for m in d["members"]]
        run_range = f"{run_label(min(run_ids, key=sort_key))}-{run_label(max(run_ids, key=sort_key))}"
        if d["has_obs"]:
            s = d["baseline_stats"]
            subtitle = f"{spec.description} | baseline bias={s['bias']:.4g} {spec.units}, RMSE={s['rmse']:.4g}, n={s['n']}"
            title = f"{CASE_LABEL} {name}: observation, baseline, members {run_range}"
        else:
            subtitle = f"{spec.description} | no mapped observation; lower panel is member - baseline"
            title = f"{CASE_LABEL} {name}: baseline, members {run_range}"
        fig.suptitle(f"{title}\n{subtitle}", x=0.02, ha="left")
        fig.autofmt_xdate(rotation=0)
        fig.subplots_adjust(top=0.86, left=0.08, right=0.98, bottom=0.10)

        safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)
        out = fig_dir / f"{safe_name}_{batch_label}_baseline_observation_experiment.png"
        fig.savefig(out, dpi=160, bbox_inches="tight")
        plt.close(fig)
        paths.append(out)
    return paths


OUT_DIR.mkdir(parents=True, exist_ok=True)
batch_label = "custom" if SELECTED_RUN_IDS is not None else f"batch{BATCH_INDEX:02d}"
summary_out = OUT_DIR / f"surface_summary_{batch_label}.csv"
SUMMARY.to_csv(summary_out, index=False)
paths = export_batch_pngs(DATA)

print(summary_out)
print(f"exported {len(paths)} figures to {OUT_DIR / batch_label}")
paths[:5]


## Optional: Export All Batches

Run this cell when you want every batch. It reloads one batch at a time to keep memory use modest.


In [ ]:
def export_all_batches(batch_size: int = BATCH_SIZE) -> pd.DataFrame:
    all_summaries = []
    n_batches = math.ceil(len(ALL_FILES) / batch_size)
    for batch_index in range(n_batches):
        batch_files = select_batch(ALL_FILES, batch_size, batch_index)
        batch_specs = discover_variable_specs(batch_files)
        batch_data, batch_summary = load_experiment_batch(batch_specs, batch_files)
        label = f"batch{batch_index:02d}"

        old_batch_index = globals().get("BATCH_INDEX")
        old_selected = globals().get("SELECTED_RUN_IDS")
        globals()["BATCH_INDEX"] = batch_index
        globals()["SELECTED_RUN_IDS"] = None
        try:
            export_batch_pngs(batch_data)
        finally:
            globals()["BATCH_INDEX"] = old_batch_index
            globals()["SELECTED_RUN_IDS"] = old_selected

        batch_summary = batch_summary.copy()
        batch_summary.insert(0, "batch_index", batch_index)
        all_summaries.append(batch_summary)
        print(f"exported {label}: runs {[run_label(x) for x in batch_files['run_id']]}")

    combined = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
    out = OUT_DIR / "surface_summary_all_batches.csv"
    combined.to_csv(out, index=False)
    print(out)
    return combined


# ALL_SUMMARY = export_all_batches()
